# Solar filament instance segmentation: participant-trained ensemble

This is the eleven-weight pipeline associated with public PQ **0.39**
(fresh replay submission 58067341). The separate 0.57 entry's private detector is unavailable.
The default path loads checksummed pretrained weights and performs new image-only
inference. Optional cells show preprocessing, training and official-label evaluation.
No archived prediction table is an inference input.

Competition data must be attached or supplied through `FILAMENT_DATA`. Source and
weights are located automatically from this directory, `FILAMENT_BUNDLE`, or an
attached Kaggle bundle. Weights included in a dataset are loaded automatically.
For a source-only release, `weights-location.json` must contain the actual public
HTTPS URLs; the bootstrap fetches and checks them automatically. The repository and release assets are currently private; anonymous public
weight fetching has not been verified.


In [ ]:
from pathlib import Path
import importlib.util, json, os, subprocess, sys, uuid
if os.environ.get('FILAMENT_BUNDLE'):
    source_bundle = Path(os.environ['FILAMENT_BUNDLE']).resolve()
elif (Path.cwd() / 'models.json').is_file():
    source_bundle = Path.cwd()
else:
    candidates = [p.parent for p in Path('/kaggle/input').rglob('models.json')
                  if (p.parent / 'owned_reproduction.py').is_file()]
    assert len(candidates) == 1, 'Set FILAMENT_BUNDLE to the source/weight bundle'
    source_bundle = candidates[0]
base_work = Path(os.environ.get('FILAMENT_OUTPUT', '/kaggle/working' if Path('/kaggle/working').is_dir()
                               else str(Path.cwd() / 'filament-output'))).resolve()
base_work.mkdir(parents=True, exist_ok=True)
spec = importlib.util.spec_from_file_location('bundle_bootstrap', source_bundle / 'bundle_bootstrap.py')
bootstrap = importlib.util.module_from_spec(spec)
spec.loader.exec_module(bootstrap)
bundle = bootstrap.prepare(source_bundle, base_work / ('downloaded-bundle-' + uuid.uuid4().hex[:10]))
manifest = json.loads((bundle / 'models.json').read_text(encoding='utf-8'))
print(manifest['role'])
print('Verified weights:', len(manifest['detectors']) + len(manifest['refiners']))


## Environment and official inputs

The verified inference environment uses Python 3.12 and the pinned CUDA 12.8
PyTorch requirements. The installation subprocess below is enabled by default;
set `FILAMENT_INSTALL=0` only when this environment is already installed.
Inference runs in a fresh subprocess so package installation cannot leave an
already-imported older neural library in use. CPU operation is slower.


In [ ]:
if os.environ.get('FILAMENT_INSTALL', '1') == '1':
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--extra-index-url',
                    'https://download.pytorch.org/whl/cu128', '-r', str(bundle / 'requirements-training.txt')], check=True)
if os.environ.get('FILAMENT_DATA'):
    data = Path(os.environ['FILAMENT_DATA']).resolve()
else:
    candidates = list(Path('/kaggle/input').rglob('test/test_images'))
    assert len(candidates) == 1, 'Set FILAMENT_DATA to MAGFiLO_1.0_Kaggle_2026'
    data = candidates[0].parent.parent
assert len(list((data / 'test/test_images').glob('*.jpeg'))) == 180
print('Official test observations:', 180)


## Optional preprocessing and training

Pretrained weights are the default. To retrain a component, set
`FILAMENT_TRAIN=1` and choose `FILAMENT_COMPONENT` from the displayed recipes,
`maskrcnn707-fourflip`, `b0a`, `b0b`, or `r34`. `FILAMENT_PREPARE_ONLY=1` executes
label/crop preparation without neural fitting. Each run writes to a new output
directory. Training uses only official TRAIN labels and never opens test pixels.

The YOLO recipes retain expert-specific examples and the union diversity arm.
Their historical time caps can yield different epoch counts on different hardware.
The neighbor member requires `FILAMENT_GONG`: exactly 1,649 SHA-verified permitted
frames and the recovered 2,683 transfer rows, with the all-site test exclusion.
It intentionally documents the 68 later quality exclusions in that historical
fit. Other components do not require extra GONG data.

The native Mask R-CNN recipe regenerates the complete compressed annotation index
and requires its exact historical SHA. Its immutable training implementation is
validated on a T4 with the pinned CUDA runtime; checkpoints resume at optimizer
boundaries through the CLI `--resume` argument. Refiners first construct conditional
soft targets from matched official experts. B0 is trained on 565 observations and
continued on all 707 for each seed; ResNet34 retains the 565-image fit. No HDJoJo
training predictions are required. Refit weights are new results, not claimed to
be bit-identical, and are not silently substituted into the fixed inference manifest.


In [ ]:
recipes = json.loads((bundle / 'training-provenance/training-recipes.json').read_text())
print('YOLO training components:', ', '.join(recipes))
RUN_TRAINING = os.environ.get('FILAMENT_TRAIN', '0') == '1'
if RUN_TRAINING:
    component = os.environ.get('FILAMENT_COMPONENT', 'long565')
    training_output = base_work / ('training-' + component + '-' + uuid.uuid4().hex[:10])
    command = [sys.executable, '-u', str(bundle / 'training_walkthrough.py'), '--data', str(data),
               '--output', str(training_output), '--component', component]
    if os.environ.get('FILAMENT_PREPARE_ONLY', '0') == '1': command.append('--prepare-only')
    subprocess.run(command, cwd=bundle, check=True)


## Neural inference and postprocessing

Seven YOLO members use each image's BatchNorm statistics without changing stored
running statistics. Native Mask R-CNN supplies a four-flip eighth detector bank.
Mask-IoU clusters allow one proposal per member, score-weight pixels and divide
confidence by all eight members, counting absence as zero. Confidence 0.2,
score-ordered ownership and a five-pixel minimum produce disjoint seeds.

Two EfficientNet-B0 refiners and a ResNet34 refiner process oriented 512 by 128
crops with grayscale and seed channels. Four crop flips are averaged; the B0 pair
and ResNet34 use the original 75/25 quantized mixture. A 24-pixel growth cap,
IoU correspondence fallback, half-weight signed-distance blending and 5% component
pruning preserve instance identities. Final RLEs are native and pixel-disjoint.

The default run processes all 180 supplied JPEGs. `FILAMENT_PROBE_STEMS` is an
explicit integration-probe override; its outputs are marked partial and must not
be presented as a complete competition export. GPU refinement uses float32 with
autocast and TF32 disabled. The exact five code cells completed the fresh default installer and all
180 images in 46.6 minutes. Anonymous public weight fetching remains unverified.
For a CPU host with at least four logical CPUs and 12 GiB free RAM, set
`FILAMENT_SHARDS=2` to compute two deterministic image partitions in parallel.
Both partitions run all eleven models from JPEGs in this invocation. Only their
new masks are merged, and full coverage is checked before a production CSV is
written. The default uses one process; GPU execution remains one process.


In [ ]:
device = os.environ.get('FILAMENT_DEVICE')
if not device:
    device = subprocess.check_output([sys.executable, '-c',
        "import torch; print('cuda:0' if torch.cuda.is_available() else 'cpu')"], text=True).strip()
output = base_work / ('predictions-' + uuid.uuid4().hex[:10])
command = [sys.executable, '-u', str(bundle / 'notebook_inference.py'), '--data', str(data),
           '--output', str(output), '--device', device, '--shards', os.environ.get('FILAMENT_SHARDS', '1')]
probe_stems = os.environ.get('FILAMENT_PROBE_STEMS', '').split()
if probe_stems: command += ['--stems', *probe_stems]
subprocess.run(command, cwd=bundle, check=True)
result = json.loads((output / 'complete.json').read_text(encoding='utf-8'))
assert result['images'] == (len(probe_stems) if probe_stems else 180)
assert result['probe_only'] == bool(probe_stems)
print(json.dumps({'partial_probe': result['probe_only'], 'images': result['images'],
                  'native_mask_audit': result['integrity'], 'csv_audit': result.get('csv_audit')}, indent=2))


## Output audit and optional official-label evaluation

`submission.csv` is newly computed from the neural chain. `predictions.json`
records every observation, including negatives; `complete.json` records execution,
integrity and output checks. Geometry validity is not an accuracy measurement.
The clean default notebook reproduced the scored replay CSV byte for byte
(SHA-256 83139e60ad3392fbdf4c06e1a287fdfe78c79dc23d7c31114934f97c3119bbff),
with 1,270 disjoint instances and one negative observation. Submission 58067341
scored public PQ 0.39. Prediction agreement measures fidelity, not accuracy.

For predictions made on an official TRAIN holdout, set `FILAMENT_VAL_PREDICTIONS`
to a JSON mapping image stem to native disjoint RLEs. The following optional cell
scores every expert record with the host's pooled all-pair IoU > 0.5 rule. It
rejects non-training stems. Only predictions of models that excluded those images
and their selected month groups constitute independent validation. The all-707
deployment models cannot claim such a holdout score. No test labels are used.


In [ ]:
validation_path = os.environ.get('FILAMENT_VAL_PREDICTIONS')
if validation_path:
    sys.path.insert(0, str(bundle))
    from solar.metrics import audit, evaluate, load_ground_truth
    predictions = json.loads(Path(validation_path).read_text())
    assert set(predictions) <= {p.stem for p in (data / 'train/train_images').glob('*.jpeg')}
    audit(predictions, set(predictions))
    truth = load_ground_truth(data / 'train/MAGFiLO_1.0_Annotations_kaggle2026_train.json', set(predictions))
    score = evaluate(truth, predictions)
    print(json.dumps({k:v for k,v in score.items() if k != 'observations'}, indent=2))
else:
    print('No independent validation predictions supplied; no accuracy score asserted.')
print('Output directory:', output)


## Provenance and release status

Original protocols, partitions and the neighbor manifest are under
`training-provenance/` and `configs/`. The newer whole-month training comparison
is a separate experiment from the historical 565-image fits. The accompanying
four-page official-template report distinguishes these cohorts and negative
results. Source is AGPL-3.0 with upstream notices and license texts included;
competition data are supplied separately under their source terms.

This documentation candidate retains the already verified notebook code
cells and all eleven weights. The clean included-weight execution is complete;
anonymous public fetching, final competition selection and the organizer form
remain pending.
It has not been certified prize eligible or presented as a winning solution.
